In [0]:
%run ./99_Config

In [0]:
%run ./98_Utilities

In [0]:
%run ./97_Logger

In [0]:
from pyspark.sql.functions import *

pipeline_start("02_Master_Data_Load")

# COMMAND ----------
REFERENCE_PATH = f"{BASE_PATH}/master_data"

datasets = sorted([
    f.name.replace("/", "")
    for f in dbutils.fs.ls(REFERENCE_PATH)
])

print("Datasets Found:", datasets)

# COMMAND ----------
total_files = 0
total_rows = 0

for dataset in datasets:

    dataset_path = f"{REFERENCE_PATH}/{dataset}"
    bronze_path = f"{BRONZE_PATH}/{dataset}"
    processed_path = f"{PROCESSED_PATH}/reference_data/{dataset}"

    print_header(f"Loading {dataset}")

    try:

        csv_files = get_csv_files(dataset_path)

        if len(csv_files) == 0:
            log_warning(f"No CSV files found for {dataset}")
            continue

        df = read_csv(dataset_path)

        rows = record_count(df)

        df = (
            df
            .withColumn("source_dataset", lit(dataset))
            .withColumn("source_file", col("_metadata.file_path"))
            .withColumn("pipeline_name", lit(PIPELINE_NAME))
            .withColumn("pipeline_run_id", lit(CURRENT_RUN_ID))
            .withColumn("load_date", current_date())
            .withColumn("load_timestamp", current_timestamp())
        )

        write_delta(df, bronze_path, "overwrite")

        create_delta_table(
            f"bronze_{dataset}",
            bronze_path
        )

        moved = move_processed_files(
            dataset_path,
            processed_path
        )

        total_files += moved
        total_rows += rows

        audit(
            notebook="02_Master_Data_Load",
            dataset=dataset,
            rows_read=rows,
            rows_written=rows,
            rejected_rows=0,
            status="SUCCESS"
        )

        log_success(
            notebook="02_Master_Data_Load",
            dataset=dataset,
            rows_read=rows,
            rows_written=rows,
            execution_time="Completed"
        )

        display_summary(dataset, rows, rows, 0)

    except Exception as ex:

        log_error(
            notebook="02_Master_Data_Load",
            dataset=dataset,
            exception=ex
        )

        audit(
            notebook="02_Master_Data_Load",
            dataset=dataset,
            rows_read=0,
            rows_written=0,
            rejected_rows=0,
            status="FAILED",
            error=str(ex)
        )

In [0]:
print("="*80)
print("MASTER DATA LOAD SUMMARY")
print("="*80)
print(f"Datasets Processed : {len(datasets)}")
print(f"Files Processed    : {total_files}")
print(f"Rows Loaded        : {total_rows}")
print("="*80)

pipeline_end("02_Master_Data_Load")